Mettez en place une interface de requêtes par LLM (avec des outils comme PandasAI, une approche prompt-to-code maison, ou des appels validés à vos propres fonctions Pandas) pour interroger votre ensemble de données ordonné en langage naturel. PandasAI vous permet de poser des questions sur les données à des LLM et intègre les LLM à d'autres composantes du pipeline, comme la visualisation de données et l'ingénierie de caractéristiques.

Vous concevrez des prompts et des outils qui permettent au LLM de comprendre des questions sur le hockey, de les convertir en requêtes structurées et de récupérer les réponses dans votre ensemble de données de la LNH.


Testez des requêtes en convertissant le langage naturel en code ou en appels de fonctions validés, par exemple : « Compare les pourcentages de buts sur tirs du poignet de Montréal et de Toronto en 2020-21 » en une requête que votre code Pandas peut exécuter. Privilégiez la réutilisation de vos fonctions d'analyse vérifiées. Si vous exécutez du code Python généré par un LLM, utilisez un environnement isolé, sans identifiants ni accès à des fichiers personnels.


Évaluez votre système sur un ensemble fixe de 8 questions d'évaluation, rédigées avant de commencer à ajuster vos prompts et jamais utilisées pour le développement des prompts. Elles doivent couvrir le filtrage, le regroupement, les pourcentages de buts, les graphiques, et au moins une question à laquelle l'ensemble de données ne permet pas de répondre (ex. sur les passes, qui ne sont pas suivies). Comparez chaque résultat à une réponse de référence que vous avez vérifiée manuellement ; un graphique est correct s'il montre les quantités, le regroupement et la saison demandés. Pour les questions sans réponse possible, le système doit indiquer qu'il ne peut pas répondre plutôt qu'inventer un résultat. Présentez un tableau indiquant, pour chaque question : si la première tentative s'est exécutée avec succès, si la réponse/le graphique final est correct, et le nombre de nouvelles tentatives. Discutez d'un succès et d'un échec ou d'une limite ; si aucun échec ne survient, testez une requête supplémentaire plus difficile et discutez-en.

8 questions d'évaluations:

-filtrage

-regroupement

-pourcentages de buts

-graphiques

-questions data ne permet pas de réponde (ex: les passes, ne sont pas suivies)

Garder les réponses de référence vérifiées manuellement

Comparer les résultats, faire un tableau indiquant le succès de chaque tentative pour chaque questions

1. What is the goal percentage done by shots of type "backhand" in the 2021 season?
    12.45%
2. Produce a figure that shows the number of shot taken on a empty net for each shot type for season 2021.
    shot_type
    backhand        53
    deflected        0
    slap             7
    snap            32
    tip-in           7
    wrap-around      1
    wrist          418
3. Produce a figure that shows the number of goals done for each type of strength for season 2016.
    strength
    even           5060
    power play      482
    shorthanded    1835
4. What is the shooting player's id for event 54 in game 411 in the playoffs for season 2019?
    8476453
5. What is the score results for event 4 in game 139 in the regular season 2023?
    1 to 4
6. What is the goal percentage done by player 8480748 in the 2022 season?
    12.28%
7. What is the most used type of pass for the 2017 season? 
    Don't have the data to answer
8. Who is the best player of the 2020 season?
    Don't have the data to answer

In [1]:
from ift6758.features.simple_visualizations import games_list
import pandas as pd

In [2]:
df2021 = games_list(2021)
df2016 = games_list(2016)
df2022 = games_list(2022)

In [4]:
#Question 1
# df2021 = games_list(2021)
backhand = df2021[df2021["shot_type"]=="backhand"]
backhand["is_goal"] = backhand["event_type"] == "goal"

shots = backhand["is_goal"].size
goals = backhand["is_goal"].sum()

print("Question 1")
print(goals/shots * 100)

#Question 2
empty_net = df2021.groupby("shot_type")["emptyNet"].agg("sum")
print("Question 2")
print(empty_net)

#Question 3
# df2016 = games_list(2016)
df2016["is_goal"] = df2016["event_type"] == "goal"
strength = df2016.groupby("strength")["is_goal"].agg("sum")
print("Question 3")
print(strength)

#Question 4
#check with ipywidget

#Question 5
#check with ipywidget

#question 6
# df2022 = games_list(2022)
player_goal = df2022[df2022["scorer_id"]==8480748]
player_shot = df2022[df2022["shooter_id"]==8480748]
print("Question 6")
print(player_goal.size/(player_goal.size+player_shot.size)*100)

Question 1
12.449293607497552
Question 2
shot_type
backhand        53
deflected        0
slap             7
snap            32
tip-in           7
wrap-around      1
wrist          418
Name: emptyNet, dtype: int64
Question 3
strength
even           5060
power play      482
shorthanded    1835
Name: is_goal, dtype: int64
Question 6
12.280701754385964


C:\Users\tiffa\AppData\Local\Temp\ipykernel_23292\3960702452.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  backhand["is_goal"] = backhand["event_type"] == "goal"


In [ ]:
import pandasai as pai

import pandasai-LiteLLM
# from pandasai_litellm.litellm import LiteLLM
from ift6758.llm

In [ ]:

# Initialize LiteLLM with your OpenAI model
# llm = LiteLLM(model="gpt-4.1-mini", api_key="YOUR_OPENAI_API_KEY")

llm = "idk for now"

pai.config.set({
    llm: llm
})

In [ ]:
#Load tidy data for each year
data = []
seasons = [2016,2017,2018,2019,2020,2021,2022,2023]
for season in seasons:
    data.append(games_list(season))

df = pd.concat(data,ignore_index=True)

df_pai = pai.DataFrame(df)

#Example
# response = df.chat("What is the average revenue by region?")
# print(response)